# Productivity of directors

In [1]:
import pandas as pd
import numpy as np

In [2]:
m = pd.read_csv('movies.csv', index_col=0)
d = pd.read_csv('directors.csv', index_col=0)

In [3]:
md = m.merge(d, how='left', left_on='director_id', right_on='id')
md.drop(columns=['id_x', 'id_y'], inplace=True)

In [4]:
md.head(3)

,budget,popularity,revenue,title,vote_average,vote_count,director_id,year,month,day,director_name,gender
0,237000000,150,2787965087,Avatar,7.2,11800,4762,2009,Dec,Thursday,James Cameron,Male
1,300000000,139,961000000,Pirates of the Caribbean: At World's End,6.9,4500,4763,2007,May,Saturday,Gore Verbinski,Male
2,245000000,107,880674609,Spectre,6.3,4466,4764,2015,Oct,Monday,Sam Mendes,Male


In [15]:
agg_md = md.groupby('director_name').agg({'year':['min','max'], 'title':'count'})
agg_md.head(3)

year       title
                              min   max count
director_name                                
Adam McKay                   2004  2015     6
Adam Shankman                2001  2012     8
Alejandro González Iñárritu  2000  2015     6

In [16]:
agg_md.columns

MultiIndex([( 'year',   'min'),
            ( 'year',   'max'),
            ('title', 'count')],
           )

In [17]:
agg_md.columns = ['year_min', 'year_max', 'title_count']

In [18]:
agg_md.head(2)

,year_min,year_max,title_count
director_name,,,
Adam McKay,2004,2015,6
Adam Shankman,2001,2012,8


In [19]:
agg_md = agg_md.reset_index()

In [20]:
agg_md['active_years'] = agg_md['year_max'] - agg_md['year_min']
agg_md['productivity'] = agg_md['title_count'] / agg_md['active_years']

In [21]:
agg_md.head(2)

,director_name,year_min,year_max,title_count,active_years,productivity
0,Adam McKay,2004,2015,6,11,0.545455
1,Adam Shankman,2001,2012,8,11,0.727273


In [23]:
agg_md.sort_values(by='productivity', ascending=False)['director_name'].iloc[0]

'Tyler Perry'

In [41]:
m['year'].nunique()

41

In [43]:
len(m['year'].value_counts())

41

In [46]:
d['gender'].value_counts(normalize=True)*100

,proportion
gender,
Male,91.299304
Female,8.700696


In [47]:
len(m[m['year'] == 2010])

63

In [48]:
md[md['year'] >= 2010]['director_name'].nunique()

163

In [49]:
md.query("title == 'Spider-Man 3'").get('director_name')

,director_name
4,Sam Raimi


In [50]:
md[md["title"] == "Spider-Man 3"]["director_name"]

,director_name
4,Sam Raimi


In [51]:
md.loc[md["title"] == "Spider-Man 3", "director_name"]

,director_name
4,Sam Raimi


In [53]:
d[d['id'] == md[md['title'] == 'Spider-Man 3']['director_id'].iloc[0]]['director_name']

,director_name
5,Sam Raimi


In [55]:
md[md["title"] == "Spider-Man 3"]["director_name"].valuecounts()

AttributeError: 'Series' object has no attribute 'valuecounts'

# In which year director was most productive

In [28]:
md.groupby(['director_name','year']).agg({'title':'count'}).reset_index().sort_values('title', ascending=False)

,director_name,year,title
1090,Robert Zemeckis,2000,2
1335,Uwe Boll,2005,2
987,Richard Donner,1992,2
582,John McTiernan,1999,2
482,James Wan,2013,2
...,...,...,...
469,James Ivory,2005,1
468,James Ivory,1995,1
467,James Ivory,1993,1
466,James Ivory,1992,1


# Melting & Pivioting

In [31]:
df = pd.read_csv('Pfizer_1.csv')
df.head(2)

,Date,Drug_Name,Parameter,1:30:00,2:30:00,3:30:00,4:30:00,5:30:00,6:30:00,7:30:00,8:30:00,9:30:00,10:30:00,11:30:00,12:30:00
0,15-10-2020,diltiazem hydrochloride,Temperature,23.0,22.0,NaN,21.0,21.0,22,23.0,21.0,22.0,20,20.0,21
1,15-10-2020,diltiazem hydrochloride,Pressure,12.0,13.0,NaN,11.0,13.0,14,16.0,16.0,24.0,18,19.0,20


In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 15 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Date       18 non-null     object 
 1   Drug_Name  18 non-null     object 
 2   Parameter  18 non-null     object 
 3   1:30:00    16 non-null     float64
 4   2:30:00    16 non-null     float64
 5   3:30:00    12 non-null     float64
 6   4:30:00    14 non-null     float64
 7   5:30:00    16 non-null     float64
 8   6:30:00    18 non-null     int64  
 9   7:30:00    16 non-null     float64
 10  8:30:00    14 non-null     float64
 11  9:30:00    16 non-null     float64
 12  10:30:00   18 non-null     int64  
 13  11:30:00   16 non-null     float64
 14  12:30:00   18 non-null     int64  
dtypes: float64(9), int64(3), object(3)
memory usage: 2.2+ KB


In [35]:
melt_data = pd.melt(df, id_vars=['Date', 'Drug_Name', 'Parameter'], var_name='Time', value_name='Reading')
melt_data.head()

,Date,Drug_Name,Parameter,Time,Reading
0,15-10-2020,diltiazem hydrochloride,Temperature,1:30:00,23.0
1,15-10-2020,diltiazem hydrochloride,Pressure,1:30:00,12.0
2,15-10-2020,docetaxel injection,Temperature,1:30:00,NaN
3,15-10-2020,docetaxel injection,Pressure,1:30:00,NaN
4,15-10-2020,ketamine hydrochloride,Temperature,1:30:00,24.0


In [40]:
pivoted_data = melt_data.pivot(index=['Date', 'Drug_Name', 'Time'], columns='Parameter', values='Reading').reset_index()
pivoted_data.head()

Parameter,Date,Drug_Name,Time,Pressure,Temperature
0,15-10-2020,diltiazem hydrochloride,10:30:00,18.0,20.0
1,15-10-2020,diltiazem hydrochloride,11:30:00,19.0,20.0
2,15-10-2020,diltiazem hydrochloride,12:30:00,20.0,21.0
3,15-10-2020,diltiazem hydrochloride,1:30:00,12.0,23.0
4,15-10-2020,diltiazem hydrochloride,2:30:00,13.0,22.0


# Binning

In [59]:
pivoted_data['Temperature'].max()

58.0

In [60]:
pivoted_data['Temperature'].min()

8.0

In [61]:
bins=[5,20,40,65]
labels=['low','medium','high']

In [63]:
pivoted_data['temp_cat'] = pd.cut(pivoted_data['Temperature'], bins=bins, labels=labels)

In [64]:
pivoted_data.head()

Parameter,Date,Drug_Name,Time,Pressure,Temperature,temp_cat
0,15-10-2020,diltiazem hydrochloride,10:30:00,18.0,20.0,low
1,15-10-2020,diltiazem hydrochloride,11:30:00,19.0,20.0,low
2,15-10-2020,diltiazem hydrochloride,12:30:00,20.0,21.0,medium
3,15-10-2020,diltiazem hydrochloride,1:30:00,12.0,23.0,medium
4,15-10-2020,diltiazem hydrochloride,2:30:00,13.0,22.0,medium


In [65]:
pivoted_data['temp_cat'].value_counts()

,count
temp_cat,
low,45
medium,38
high,12


In [66]:
type(np.nan)

float

In [67]:
type(None)

NoneType

In [76]:
type(pd.Series(['subham','None', np.nan]).iloc[1])

str

In [81]:
df.isna().sum()

,0
Date,0
Drug_Name,0
Parameter,0
1:30:00,2
2:30:00,2
3:30:00,6
4:30:00,4
5:30:00,2
6:30:00,0
7:30:00,2


In [78]:
df.isnull()

,Date,Drug_Name,Parameter,1:30:00,2:30:00,3:30:00,4:30:00,5:30:00,6:30:00,7:30:00,8:30:00,9:30:00,10:30:00,11:30:00,12:30:00
0,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False
2,False,False,False,True,False,False,True,False,False,True,True,False,False,False,False
3,False,False,False,True,False,False,True,False,False,True,True,False,False,False,False
4,False,False,False,False,True,True,False,True,False,False,False,False,False,False,False
5,False,False,False,False,True,True,False,True,False,False,False,False,False,False,False
6,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
7,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
8,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False
9,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False


In [83]:
df.shape

(18, 15)

In [86]:
df.isna().sum(axis=0)

,0
Date,0
Drug_Name,0
Parameter,0
1:30:00,2
2:30:00,2
3:30:00,6
4:30:00,4
5:30:00,2
6:30:00,0
7:30:00,2


In [ ]:
df.dropna()